# Pipeline DINO + MedSAM amélioré — V2 (cascade pancréas→tumor + pad 8% + CC)

**Identique à `msd_implementation/notebooks/dino_medsam_cascade/improved_pipeline.ipynb`** (UNet, gating, calibration, ablation 6 configs, dual-prompt, VLM…) mais la **baseline DINO de référence** est remplacée par la cascade `baseline_v2 best config` :
- prompt `"pancreas . tumor ."` + filtrage label
- box pancréas DINO (`label==0`, score>0.30) + marge 20 px
- tumor (`label==1`, score>0.01) doit overlap ≥ 10 % avec la box pancréas
- pad 8 % de la box tumor avant MedSAM
- composante connexe max sur le masque

Le UNet pancréas reste utilisé pour les configs `+G1*` afin de garder la comparaison historique. Cible : confirmer que l'ablation gardée donne ≥ 0.59 DICE sur le baseline (vs 0.447 dans la version originale).

## 0. Setup Colab (clone + deps + symlinks Drive)

Premier lancement : ~2 min (install deps) + monter Drive. Relancer la cellule est idempotent.

In [ ]:
# === Colab bootstrap ================================================
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, sys, subprocess
    REPO = '/content/INF8225_Projet'
    if not os.path.isdir(REPO):
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'main',
            'https://github.com/Azcatchi17/INF8225_Projet.git', REPO])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)

    from colab.setup import setup
    setup()
else:
    print('Local run — skipping Colab setup. Make sure data/ and work_dirs/ are populated.')

## 1. Imports & config override

Après `setup()`, les chemins dans `config.py` pointent vers les dossiers Drive via symlink. On charge tout.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch

from msd_implementation.pipelines.dino_medsam_gemini import (
    config, pancreas_roi, gating, calibrate, eval_improved, grounding_dino as gd, medsam,
)
from msd_implementation.pipelines.dino_medsam_gemini import metrics as M
from experiments.baseline_v2 import dino_v2, eval_v2, postprocess

# === V2 best config (cf. baseline_v2.ipynb §6.2) =====================
BEST_CFG = dict(
    text='pancreas . tumor .',
    pancreas_thr=0.30,
    tumor_thr=0.01,
    pancreas_margin_px=20,
    min_overlap_ratio=0.10,
    pad_frac=0.08,
    keep_largest=True,
    min_area_px=0,
    max_area_frac=1.0,
)

def _baseline_v2_oneshot(img_path: str):
    """Cascade DINO (pancréas→tumor) + pad 8% + CC max — remplace eval_improved._baseline_oneshot."""
    image_np = medsam.load_image(img_path)
    H, W = image_np.shape[:2]
    box = dino_v2.cascade_select_tumor_box(
        img_path, text=BEST_CFG['text'],
        pancreas_thr=BEST_CFG['pancreas_thr'], tumor_thr=BEST_CFG['tumor_thr'],
        pancreas_margin_px=BEST_CFG['pancreas_margin_px'],
        min_overlap_ratio=BEST_CFG['min_overlap_ratio'],
    )
    if box is None:
        return np.zeros((H, W), np.uint8), 0.0
    embed = medsam.encode_image(image_np)
    padded = postprocess.pad_box(box.xyxy, BEST_CFG['pad_frac'], H, W)
    mask = medsam.segment(embed, H=H, W=W, box=list(padded))
    if BEST_CFG['keep_largest']:
        mask = postprocess.keep_largest_cc(mask)
    return mask, float(box.score)

print('CUDA available:', torch.cuda.is_available())
print('MSD root :', config.MSD_ROOT, config.MSD_ROOT.exists())
print('DINO cfg :', config.DINO_CONFIG, config.DINO_CONFIG.exists())
print('DINO ckpt:', config.DINO_CHECKPOINT, config.DINO_CHECKPOINT.exists())
print('MedSAM   :', config.MEDSAM_CHECKPOINT, config.MEDSAM_CHECKPOINT.exists())
print('UNet ckpt:', config.PANCREAS_CKPT, config.PANCREAS_CKPT.exists())
print('\nV2 BEST_CFG :')
for k, v in BEST_CFG.items():
    print(f'  {k:22s} = {v}')

## 2. Entraîner (ou charger) le UNet pancréas

Cible : `mask==1` dans les masques MSD. 4-level UNet (~1.9 M params à base=32).

**Hyperparams v2** (après ablation v1 décevante à DICE 0.41 / sens 0.73) :
- Input **384 px** (au lieu de 256)
- **30 epochs** (au lieu de 15)
- **Tversky loss** avec `β=0.7, α=0.3` → 2× pénalité FN vs FP → ROI plus généreuse, privilégie le recall (ce qu'on veut pour un gate)
- Augmentations : flip horizontal + bruit gaussien léger

- Premier run : entraîne ~10-15 min sur T4 et sauvegarde dans `work_dirs/pancreas_unet/best.pt` (Drive)
- Runs suivants : charge le checkpoint existant

Pour ré-entraîner, laisse `RETRAIN_UNET = True` ou supprime le `.pt`.

In [ ]:
RETRAIN_UNET = False  # forcer le ré-entraînement avec Tversky + 384 px

# Hyperparams v2: input 384, 30 epochs, Tversky (β=0.7 → 2× pénalité FN, ROI plus généreuse)
if RETRAIN_UNET or not config.PANCREAS_CKPT.exists():
    print(f'Training pancreas UNet → {config.PANCREAS_CKPT}')
    train_info = pancreas_roi.train_pancreas_unet(
        train_json=config.MSD_ROOT / 'train.json',
        val_json=config.MSD_ROOT / 'val.json',
        data_root=config.MSD_ROOT,
        epochs=30,
        batch_size=6,            # 384 px + base=32 ⇒ ~3.5 GB VRAM @ bs=6 sur T4
        lr=1e-3,
        img_size=384,
        loss_name='tversky',
        tversky_beta=0.7,
        base_channels=32,
        save_path=config.PANCREAS_CKPT,
    )
    print('best val dice =', train_info.best_val_dice)
    # Drop the in-memory model so the fresh weights get loaded on next call
    pancreas_roi.reset_cache()
else:
    print(f'Loading cached pancreas UNet from {config.PANCREAS_CKPT}')
    unet = pancreas_roi.get_pancreas_unet()
    print('val dice at save time =', unet['val_dice'], ' img_size =', unet['img_size'])

## 3. Sanity check qualitatif : le ROI pancréas tient la route ?

On prend 6 slices du split test et on visualise le masque UNet + la bbox dilatée.

In [ ]:
with open(config.MSD_TEST_JSON) as f:
    test_meta = json.load(f)

# Mix tumor-present / tumor-absent
def _has_tumor(file_name):
    m = np.array(Image.open(config.MSD_ROOT / file_name.replace('/images/', '/masks/')))
    if m.ndim == 3: m = m[..., 0]
    return int((m == 2).sum()) > 25

present = [i for i in test_meta['images'] if _has_tumor(i['file_name'])][:3]
absent  = [i for i in test_meta['images'] if not _has_tumor(i['file_name'])][:3]
picks = present + absent

fig, axes = plt.subplots(len(picks), 3, figsize=(12, 4 * len(picks)))
for row, info in enumerate(picks):
    img_path = config.MSD_ROOT / info['file_name']
    image = medsam.load_image(str(img_path))
    pmask, pbbox = pancreas_roi.infer_pancreas(image, dilation_px=config.PANCREAS_DILATION_PX)

    axes[row, 0].imshow(image, cmap='gray')
    axes[row, 0].set_title(f"CT  — {Path(info['file_name']).name}\nhas_tumor={_has_tumor(info['file_name'])}")
    axes[row, 0].axis('off')

    axes[row, 1].imshow(image, cmap='gray')
    axes[row, 1].imshow(np.ma.masked_where(pmask == 0, pmask), cmap='Greens', alpha=0.4)
    if pbbox is not None:
        x1, y1, x2, y2 = pbbox
        axes[row, 1].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='lime', lw=2))
    axes[row, 1].set_title(f"G1: pancreas ROI  (area={int(pmask.sum())} px)")
    axes[row, 1].axis('off')

    gt_mask = np.array(Image.open(config.MSD_ROOT / info['file_name'].replace('/images/', '/masks/')))
    if gt_mask.ndim == 3: gt_mask = gt_mask[..., 0]
    axes[row, 2].imshow(image, cmap='gray')
    axes[row, 2].imshow(np.ma.masked_where(gt_mask != 1, gt_mask), cmap='Greens', alpha=0.5)
    axes[row, 2].imshow(np.ma.masked_where(gt_mask != 2, gt_mask), cmap='autumn', alpha=0.7)
    axes[row, 2].set_title("GT: pancreas (green) + tumor (red)")
    axes[row, 2].axis('off')
plt.tight_layout(); plt.show()

## 4. Calibrer le seuil τ sur le split val

On exécute **DINO + ROI pancréas** sur les 100 images val, on récupère le score max qui survit au filtre, puis on balaie τ et on choisit celui qui maximise F1 (tumor/no-tumor comme classification binaire).

Le résultat écrase `config.TUMOR_DIFF_THRESHOLD` pour la suite de la session.

In [ ]:
calib = calibrate.calibrate_threshold(
    val_json=config.MSD_ROOT / 'val.json',
    use_pancreas_roi=True,
    n=None,
    n_thresholds=60,
)
print(f'optimal τ          = {calib.threshold:.4f}')
print(f'F1 @τ              = {calib.f1:.3f}')
print(f'sensitivity / spec = {calib.sensitivity:.3f} / {calib.specificity:.3f}')
print(f'TP/FP/TN/FN        = {calib.tp}/{calib.fp}/{calib.tn}/{calib.fn}')

# Apply to config for the rest of the notebook
config.TUMOR_DIFF_THRESHOLD = calib.threshold

# Persist to Drive so other runs can re-use it
calibrate.save_calibration(calib, config.AGENT_RESULTS_DIR / 'calibration.json')
print('saved calibration →', config.AGENT_RESULTS_DIR / 'calibration.json')

In [ ]:
# Score distribution + F1 vs τ curve
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bins = np.linspace(0, max(max(calib.scores_pos or [0.01]), max(calib.scores_neg or [0.01])) * 1.05, 30)
axes[0].hist(calib.scores_neg, bins=bins, alpha=0.55, label=f'no-tumor (n={len(calib.scores_neg)})', color='tab:blue')
axes[0].hist(calib.scores_pos, bins=bins, alpha=0.55, label=f'tumor (n={len(calib.scores_pos)})', color='tab:red')
axes[0].axvline(calib.threshold, color='k', ls='--', label=f'τ={calib.threshold:.3f}')
axes[0].set_xlabel('Gated DINO best-box score'); axes[0].set_ylabel('count')
axes[0].set_title('Score distribution (val)')
axes[0].legend()

axes[1].plot(calib.thresholds, calib.f1_curve, label='F1')
axes[1].plot(calib.thresholds, calib.sens_curve, label='sensitivity', ls='--')
axes[1].plot(calib.thresholds, calib.spec_curve, label='specificity', ls=':')
axes[1].axvline(calib.threshold, color='k', ls='--', alpha=0.5)
axes[1].set_xlabel('τ'); axes[1].set_ylabel('metric'); axes[1].set_title('Metric vs threshold')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Démo visuelle gatée vs baseline

On prend les 3 tumeurs + 3 non-tumeurs de la section 3, et on compare :
- **Baseline** : DINO → top-1 → MedSAM
- **Gated** : G1 + G2 + G3 (seuil calibré)

In [ ]:
fig, axes = plt.subplots(len(picks), 3, figsize=(14, 4 * len(picks)))

for row, info in enumerate(picks):
    img_path = config.MSD_ROOT / info['file_name']
    image = medsam.load_image(str(img_path))
    has_tumor = _has_tumor(info['file_name'])
    gt_mask = np.array(Image.open(config.MSD_ROOT / info['file_name'].replace('/images/', '/masks/')))
    if gt_mask.ndim == 3: gt_mask = gt_mask[..., 0]
    gt_tumor = (gt_mask == 2).astype(np.uint8)

    # Baseline V2 (cascade + pad + CC)
    bmask, bscore = _baseline_v2_oneshot(str(img_path))

    # Gated (UNet ROI + threshold)
    res = gating.infer_gated(str(img_path), use_pancreas_roi=True, use_vlm_verify=False,
                              use_dual_prompt=False, score_threshold=calib.threshold)

    # (a) CT + GT tumor
    axes[row, 0].imshow(image, cmap='gray')
    axes[row, 0].imshow(np.ma.masked_where(gt_tumor == 0, gt_tumor), cmap='autumn', alpha=0.6)
    axes[row, 0].set_title(f"GT  — has_tumor={has_tumor}"); axes[row, 0].axis('off')

    # (b) Baseline V2 prediction
    axes[row, 1].imshow(image, cmap='gray')
    if bmask.sum() > 0:
        axes[row, 1].imshow(np.ma.masked_where(bmask == 0, bmask), cmap='autumn', alpha=0.5)
    from msd_implementation.pipelines.dino_medsam_gemini.metrics import dice as _dice
    axes[row, 1].set_title(f"baseline V2  dice={_dice(bmask, gt_tumor):.2f}")
    axes[row, 1].axis('off')

    # (c) Gated prediction
    axes[row, 2].imshow(image, cmap='gray')
    if res.pancreas_bbox is not None:
        x1, y1, x2, y2 = res.pancreas_bbox
        axes[row, 2].add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='lime', lw=1, alpha=0.5))
    if res.selected_box is not None:
        x1, y1, x2, y2 = res.selected_box
        axes[row, 2].add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='yellow', lw=2))
    if res.mask.sum() > 0:
        axes[row, 2].imshow(np.ma.masked_where(res.mask == 0, res.mask), cmap='autumn', alpha=0.5)
    axes[row, 2].set_title(f"gated {res.decision}  dice={_dice(res.mask, gt_tumor):.2f}")
    axes[row, 2].axis('off')

plt.tight_layout(); plt.show()

## 6. Ablation quantitative sur test

On évalue jusqu'à 6 configurations sur les 100 images test :

| Config                   | G1 | G3 τ     | Dual-prompt | VLM |
|--------------------------|----|----------|-------------|-----|
| `baseline`               | ✗  | -        | ✗           | ✗   |
| `+G1 pancreas ROI`       | ✓  | 0        | ✗           | ✗   |
| `+G1+G3 threshold`       | ✓  | calibré  | ✗           | ✗   |
| `+G1+G3+dual_prompt`     | ✓  | calibré  | ✓           | ✗   |
| `+G1+G3+VLM`             | ✓  | calibré  | ✗           | ✓   |
| `+G1+G3+dual+VLM`        | ✓  | calibré  | ✓           | ✓   |

- **Dual-prompt** : relance DINO avec `["normal pancreas tissue", "bowel gas", "abdominal organ"]` et soustrait `0.5 · max_distracteur` du score tumor → tue les candidats qui ressemblent autant à du distracteur qu'à de la tumeur.
- **VLM** : Gemini regarde le crop du candidat top-1 survivant ; si son verdict est `is_tumor=False`, on rejette. ~100 appels max par config activée. Nécessite `GEMINI_API_KEY` dans les Secrets Colab.

Tu peux désactiver individuellement via les flags `INCLUDE_DUAL_PROMPT`, `INCLUDE_VLM`, `INCLUDE_BOTH` dans la cellule suivante.

In [ ]:
INCLUDE_DUAL_PROMPT = True    # relance DINO avec ["normal pancreas tissue", "bowel gas", "abdominal organ"] et soustrait le max distracteur
INCLUDE_VLM         = True    # Gemini vérifie sémantiquement le crop top-1 — nécessite GEMINI_API_KEY
INCLUDE_BOTH        = True    # dual-prompt ET VLM combinés
N_TEST = None                  # None = les 100 images du split test

import json as _json
from datetime import datetime as _dt

def _gated(name, **kw):
    print(f'\n=== {name} ===')
    return eval_improved.run_gated(
        coco_json=config.MSD_TEST_JSON,
        n=N_TEST,
        score_threshold=calib.threshold,
        pancreas_ckpt=str(config.PANCREAS_CKPT),
        **kw,
    )

ablation = {}
print('=== baseline V2 (cascade + pad 8% + CC) ===')
ablation['baseline V2'] = eval_v2.run_baseline_v2(
    coco_json=config.MSD_TEST_JSON, n=N_TEST,
    progress_desc='baseline V2',
    **BEST_CFG,
)
ablation['+G1 pancreas ROI'] = _gated('+G1 pancreas ROI',
    use_pancreas_roi=True, use_vlm_verify=False, use_dual_prompt=False)
ablation['+G1+G3 threshold'] = _gated('+G1+G3 threshold',
    use_pancreas_roi=True, use_vlm_verify=False, use_dual_prompt=False)

if INCLUDE_DUAL_PROMPT:
    ablation['+G1+G3+dual_prompt'] = _gated('+G1+G3+dual_prompt',
        use_pancreas_roi=True, use_vlm_verify=False, use_dual_prompt=True)

if INCLUDE_VLM:
    ablation['+G1+G3+VLM'] = _gated('+G1+G3+VLM',
        use_pancreas_roi=True, use_vlm_verify=True, use_dual_prompt=False)

if INCLUDE_BOTH:
    ablation['+G1+G3+dual+VLM'] = _gated('+G1+G3+dual+VLM',
        use_pancreas_roi=True, use_vlm_verify=True, use_dual_prompt=True)

# Persist to Drive
save_dir = config.AGENT_RESULTS_DIR
save_dir.mkdir(parents=True, exist_ok=True)
ts = _dt.now().strftime('%Y-%m-%dT%H-%M-%S')
for name, (df, summ) in ablation.items():
    safe = name.replace(' ', '_').replace('+', 'p')
    df.to_csv(save_dir / f'ablation_v2_{ts}_{safe}.csv', index=False)
with open(save_dir / f'ablation_v2_{ts}_summary.json', 'w') as f:
    _json.dump({name: summ.to_dict() for name, (_, summ) in ablation.items()}, f, indent=2)
print(f'\nsaved → {save_dir / f"ablation_v2_{ts}_summary.json"}')

summary_rows = []
for name, (df, summ) in ablation.items():
    summary_rows.append({'config': name, **summ.to_dict()})
summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
# Metric comparison bars — dynamique, supporte n configs
fig, axes = plt.subplots(1, 3, figsize=(max(15, 2.5 * len(summary_df)), 4.5))

x = np.arange(len(summary_df))
rot = 25 if len(summary_df) <= 5 else 40
axes[0].bar(x, summary_df['sensitivity'], color='tab:blue')
axes[0].set_ylim(0, 1); axes[0].set_title('Sensitivity (tumor recall)')
axes[0].set_xticks(x); axes[0].set_xticklabels(summary_df['config'], rotation=rot, ha='right')

axes[1].bar(x, summary_df['specificity'], color='tab:green')
axes[1].set_ylim(0, 1); axes[1].set_title('Specificity (no-tumor rejection)')
axes[1].set_xticks(x); axes[1].set_xticklabels(summary_df['config'], rotation=rot, ha='right')

axes[2].bar(x, summary_df['dice_tumor_present_mean'], color='tab:red')
axes[2].set_ylim(0, 1); axes[2].set_title('DICE on tumor-present images')
axes[2].set_xticks(x); axes[2].set_xticklabels(summary_df['config'], rotation=rot, ha='right')

for ax, col in zip(axes, ['sensitivity', 'specificity', 'dice_tumor_present_mean']):
    for xi, v in enumerate(summary_df[col]):
        ax.text(xi, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Per-image DICE comparison: baseline V2 vs best-F1 gated config
best_name = max(
    (name for name in ablation if name != 'baseline V2'),
    key=lambda n: ablation[n][1].f1,
)
print(f'best gated config by F1: {best_name}')

baseline_df = ablation['baseline V2'][0].set_index('file_name')
best_df     = ablation[best_name][0].set_index('file_name')
cmp = baseline_df[['has_tumor', 'dice']].rename(columns={'dice': 'dice_baseline'}).join(
    best_df[['dice']].rename(columns={'dice': 'dice_best'})
).reset_index()
cmp['dice_delta'] = cmp['dice_best'] - cmp['dice_baseline']
cmp = cmp.sort_values('has_tumor', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 4))
idx = np.arange(len(cmp))
ax.bar(idx - 0.2, cmp['dice_baseline'], width=0.4, label='baseline V2', color='tab:gray')
ax.bar(idx + 0.2, cmp['dice_best'], width=0.4, label=best_name, color='tab:red')
ax.axvline(cmp['has_tumor'].sum() - 0.5, color='k', ls=':', alpha=0.5)
ax.set_xlabel('images (tumor-present à gauche de la ligne)')
ax.set_ylabel('DICE')
ax.set_title(f"Per-image DICE  baseline V2={cmp['dice_baseline'].mean():.3f}  {best_name}={cmp['dice_best'].mean():.3f}")
ax.legend(); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrices per configuration — grid layout pour n configs
from itertools import product
n_cfg = len(ablation)
ncols = min(n_cfg, 3)
nrows = (n_cfg + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.atleast_2d(axes).ravel()
for idx, (name, (_, summ)) in enumerate(ablation.items()):
    ax = axes[idx]
    M_ = np.array([[summ.tn, summ.fp], [summ.fn, summ.tp]])
    ax.imshow(M_, cmap='Blues')
    for i, j in product(range(2), range(2)):
        ax.text(j, i, str(M_[i, j]), ha='center', va='center', fontsize=14,
                color='white' if M_[i, j] > M_.max() / 2 else 'black')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['pred no-tumor', 'pred tumor'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['GT no-tumor', 'GT tumor'])
    ax.set_title(f"{name}\nsens={summ.sensitivity:.2f}  spec={summ.specificity:.2f}  F1={summ.f1:.2f}")
for idx in range(n_cfg, len(axes)):
    axes[idx].axis('off')
plt.tight_layout(); plt.show()

## 7. Cas où la gating fait la différence

On extrait les 4 images où `dice_gated - dice_baseline` est maximal (gating rejette un faux positif, ou recale la box).

In [ ]:
# Visualisation des 4 images où la config gagnante apporte le plus de DICE vs baseline V2
picks_delta = cmp.nlargest(4, 'dice_delta')['file_name'].tolist()

# Match best config name to gating flags
BEST_FLAGS = {
    '+G1 pancreas ROI':    dict(use_pancreas_roi=True, use_vlm_verify=False, use_dual_prompt=False),
    '+G1+G3 threshold':    dict(use_pancreas_roi=True, use_vlm_verify=False, use_dual_prompt=False),
    '+G1+G3+dual_prompt':  dict(use_pancreas_roi=True, use_vlm_verify=False, use_dual_prompt=True),
    '+G1+G3+VLM':          dict(use_pancreas_roi=True, use_vlm_verify=True,  use_dual_prompt=False),
    '+G1+G3+dual+VLM':     dict(use_pancreas_roi=True, use_vlm_verify=True,  use_dual_prompt=True),
}
flags = BEST_FLAGS.get(best_name, dict(use_pancreas_roi=True, use_vlm_verify=False, use_dual_prompt=False))

fig, axes = plt.subplots(len(picks_delta), 3, figsize=(13, 4 * len(picks_delta)))
if len(picks_delta) == 1:
    axes = axes[None, :]
for row, fn in enumerate(picks_delta):
    img_path = config.MSD_ROOT / fn
    image = medsam.load_image(str(img_path))
    gt_mask = np.array(Image.open(config.MSD_ROOT / fn.replace('/images/', '/masks/')))
    if gt_mask.ndim == 3: gt_mask = gt_mask[..., 0]
    gt_tumor = (gt_mask == 2).astype(np.uint8)

    bmask, _ = _baseline_v2_oneshot(str(img_path))
    res = gating.infer_gated(str(img_path), score_threshold=calib.threshold, **flags)

    axes[row, 0].imshow(image, cmap='gray')
    axes[row, 0].imshow(np.ma.masked_where(gt_tumor == 0, gt_tumor), cmap='autumn', alpha=0.6)
    axes[row, 0].set_title(f"GT — {Path(fn).name}\nhas_tumor={gt_tumor.sum() > 0}"); axes[row, 0].axis('off')

    axes[row, 1].imshow(image, cmap='gray')
    if bmask.sum() > 0:
        axes[row, 1].imshow(np.ma.masked_where(bmask == 0, bmask), cmap='autumn', alpha=0.5)
    axes[row, 1].set_title(f'baseline V2  dice={_dice(bmask, gt_tumor):.2f}'); axes[row, 1].axis('off')

    axes[row, 2].imshow(image, cmap='gray')
    if res.pancreas_bbox is not None:
        x1, y1, x2, y2 = res.pancreas_bbox
        axes[row, 2].add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='lime', lw=1, alpha=0.5))
    if res.mask.sum() > 0:
        axes[row, 2].imshow(np.ma.masked_where(res.mask == 0, res.mask), cmap='autumn', alpha=0.5)
    axes[row, 2].set_title(f'{best_name}  {res.decision}  dice={_dice(res.mask, gt_tumor):.2f}')
    axes[row, 2].axis('off')
plt.tight_layout(); plt.show()

## 8. (Optionnel) Gating branché sur l'agent Gemini

Quand la gate accepte une box, on la passe au `run_agent` habituel pour bénéficier du raffinement itératif. Quand la gate rejette, on économise tous les appels Gemini + MedSAM.

In [ ]:
# Uncomment to run. Requires GEMINI_API_KEY in Colab Secrets.
from msd_implementation.pipelines.dino_medsam_gemini.agent import run_agent
img_info = picks[0]  # first tumor image
state = run_agent(
    str(config.MSD_ROOT / img_info['file_name']),
    user_text='find the pancreatic tumor',
    use_gating=True,
    gating_kwargs={'score_threshold': calib.threshold, 'use_vlm_verify': False},
    persist=False,
)
print('stop_reason:', state.stop_reason)
print('# iterations:', len(state.iterations))
print('best iter dice:', state.best_iter().dice_vs_gt if state.best_iter() else None)

## 9. Résumé — ce qui a bougé

| Problème initial | Solution appliquée | Porte |
|---|---|---|
| DINO hallucine hors pancréas | Filtre géométrique sur ROI UNet | G1+G2 |
| Negative prompts `no tumor` dégradent | Remplacés par score différentiel dual-prompt (optionnel) + vérif VLM | G2.5+G3 |
| Images sans tumeur mal gérées | Seuil τ calibré sur val → masque vide honnête | G3 |
| MedSAM segmente n'importe quoi | Box clippée à la bbox pancréas, mask AND-é avec ROI | G1 |
| Classifieur amont inefficace | Remplacé par la combinaison ROI + τ (signal global ineffficace, signal local gate-filtré efficace) | architecture |

**Checkpoints persistés dans Drive :**
- `work_dirs/pancreas_unet/best.pt` (UNet)
- `data/agent_results_msd/calibration.json` (τ et courbes)
- `data/agent_results_msd/ablation_*_*.csv` + `ablation_*_summary.json`